Setup and Load Datasets

In [1]:
!pip install imbalanced-learn

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    average_precision_score
)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [3]:
data1 = pd.read_csv('Epileptic Seizure Recognition.csv')

data2 = pd.read_csv('BEED_Data.csv')

data3 = pd.read_csv('eeg_seizure_analysis.csv')

Dataset Preparation

In [4]:
data1 = data1.loc[:, ~data1.columns.str.contains('^Unnamed')]

In [5]:
data1['y'] = data1['y'].apply(
    lambda x: 1 if x == 1 else 0
)

In [6]:
print(data1.shape)
print(data2.shape)
print(data3.shape)

(11500, 179)
(8000, 17)
(8000, 17)


Train-Test Split

In [8]:
datasets = []

for df in [data1,data2,data3]:

    X = df.drop('y', axis=1)

    y = df['y']

    X_train,X_test,y_train,y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    datasets.append(
        (X_train,X_test,y_train,y_test)
    )

**Pipeline A**

In [9]:
pipelineA = []

for X_train,X_test,y_train,y_test in datasets:

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)

    X_test_scaled = scaler.transform(X_test)

    k = min(50, X_train.shape[1])

    selector = SelectKBest(
        score_func=f_classif,
        k=k
    )

    X_train_A = selector.fit_transform(
        X_train_scaled,
        y_train
    )

    X_test_A = selector.transform(
        X_test_scaled
    )

    pipelineA.append(
        (X_train_A,X_test_A,y_train,y_test)
    )

**Pipeline B**

In [10]:
pipelineB = []

for X_train,X_test,y_train,y_test in datasets:

    scaler = MinMaxScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_test_scaled = scaler.transform(
        X_test
    )

    pca = PCA(
        n_components=0.95
    )

    X_train_B = pca.fit_transform(
        X_train_scaled
    )

    X_test_B = pca.transform(
        X_test_scaled
    )

    pipelineB.append(
        (X_train_B,X_test_B,y_train,y_test)
    )

**Models**

In [11]:
l1 = LogisticRegression(
    penalty='l1',
    solver='saga',
    max_iter=5000
)

l2 = LogisticRegression(
    penalty='l2',
    max_iter=5000
)

elastic = LogisticRegression(
    penalty='elasticnet',
    l1_ratio=0.5,
    solver='saga',
    max_iter=5000
)

models = {
    "L1": l1,
    "L2": l2,
    "Elastic Net": elastic
}

**Run All Experiments**

In [12]:
results = []

In [13]:
for idx, (X_train,X_test,y_train,y_test) in enumerate(pipelineA):

    for name, model in models.items():

        model.fit(X_train,y_train)

        pred = model.predict(X_test)

        acc = accuracy_score(y_test,pred)

        f1 = f1_score(
            y_test,
            pred,
            average='weighted'
        )

        try:

            prob = model.predict_proba(X_test)

            if len(np.unique(y_test)) == 2:

                pr_auc = average_precision_score(
                    y_test,
                    prob[:,1]
                )

            else:

                pr_auc = np.nan

        except:

            pr_auc = np.nan

        results.append([
            f"Dataset{idx+1}",
            "Pipeline A",
            name,
            "None",
            acc,
            f1,
            pr_auc
        ])

In [14]:
for idx, (X_train,X_test,y_train,y_test) in enumerate(pipelineB):

    for name, model in models.items():

        model.fit(X_train,y_train)

        pred = model.predict(X_test)

        acc = accuracy_score(y_test,pred)

        f1 = f1_score(
            y_test,
            pred,
            average='weighted'
        )

        try:

            prob = model.predict_proba(X_test)

            if len(np.unique(y_test)) == 2:

                pr_auc = average_precision_score(
                    y_test,
                    prob[:,1]
                )

            else:

                pr_auc = np.nan

        except:

            pr_auc = np.nan

        results.append([
            f"Dataset{idx+1}",
            "Pipeline B",
            name,
            "None",
            acc,
            f1,
            pr_auc
        ])

**Final Comparison Table**

In [15]:
final_results = pd.DataFrame(
    results,
    columns=[
        "Dataset",
        "Pipeline",
        "Regularization",
        "Imbalance",
        "Accuracy",
        "F1",
        "PR_AUC"
    ]
)

display(final_results)

,Dataset,Pipeline,Regularization,Imbalance,Accuracy,F1,PR_AUC
0,Dataset1,Pipeline A,L1,None,0.810870,0.736050,0.474846
1,Dataset1,Pipeline A,L2,None,0.810435,0.735092,0.475291
2,Dataset1,Pipeline A,Elastic Net,None,0.810435,0.735092,0.475442
3,Dataset2,Pipeline A,L1,None,0.457500,0.464305,NaN
4,Dataset2,Pipeline A,L2,None,0.454375,0.460894,NaN
5,Dataset2,Pipeline A,Elastic Net,None,0.455625,0.462379,NaN
6,Dataset3,Pipeline A,L1,None,0.457500,0.464305,NaN
7,Dataset3,Pipeline A,L2,None,0.454375,0.460894,NaN
8,Dataset3,Pipeline A,Elastic Net,None,0.455625,0.462379,NaN
9,Dataset1,Pipeline B,L1,None,0.803043,0.718311,0.458675


**Save Results**

In [16]:
final_results.to_csv(
    "final_results.csv",
    index=False
)

**Sparsity Analysis**

In [21]:
X_train_l1, _, y_train_l1, _ = pipelineA[0]

for name, model in models.items():

    model.fit(X_train_l1, y_train_l1)

    coef = model.coef_

    zero_weights = np.sum(coef == 0)

    total_weights = coef.size

    sparsity = (zero_weights / total_weights) * 100

    print("\n", name)
    print("Zero Weights:", zero_weights)
    print("Total Weights:", total_weights)
    print("Sparsity:", sparsity)


 L1
Zero Weights: 4
Total Weights: 50
Sparsity: 8.0

 L2
Zero Weights: 0
Total Weights: 50
Sparsity: 0.0

 Elastic Net
Zero Weights: 2
Total Weights: 50
Sparsity: 4.0


In [20]:
underfit = LogisticRegression(
    C=0.001,
    penalty='l2',
    max_iter=5000
)

underfit.fit(X_train_A, y_train)

train_acc = underfit.score(
    X_train_A,
    y_train
)

test_acc = underfit.score(
    X_test_A,
    y_test
)

print(train_acc)
print(test_acc)

0.50015625
0.494375


In [22]:
overfit = LogisticRegression(
    C=100000,
    penalty='l2',
    max_iter=5000
)

overfit.fit(X_train_A, y_train)

train_acc = overfit.score(
    X_train_A,
    y_train
)

test_acc = overfit.score(
    X_test_A,
    y_test
)

print(train_acc)
print(test_acc)

0.48171875
0.454375


In [27]:
print(
    accuracy_score(y_test,pred)
)

print(
    f1_score(
        y_test,
        pred,
        average='weighted'
    )
)

print(
    average_precision_score(
        y_test,
        prob,
        average='weighted'
    )
)

0.433125
0.43114650381613545
0.38617487947403006
